# MediKiosk - Fine-Tuning Llama 3.1 8B with Unsloth / QLoRA
This notebook fine-tunes **Meta-Llama-3.1-8B-Instruct** using Unsloth (2x faster, 70% less VRAM) on the curated **AyurGenixAI** clinical dataset for structured Ayurvedic clinical case extraction in MediKiosk.

In [ ]:
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers trl peft accelerate bitsandbytes datasets triton


In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None  # Auto detects Float16 for Tesla T4
load_in_4bit = True  # 4-bit quantization via bitsandbytes

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Meta-Llama-3.1-8B-Instruct",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)


In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0, # Optimized to 0 by Unsloth for speed
    bias = "none",
    use_gradient_checkpointing = "unsloth", # 30% less VRAM
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)
print("LoRA adapters successfully attached.")


In [ ]:
from datasets import load_dataset

# Load uploaded train.jsonl (contains 402 clean Ayurvedic training pairs)
dataset = load_dataset("json", data_files={"train": "train.jsonl"}, split="train")

llama3_prompt = """<|start_header_id|>system<|end_header_id|>

{instruction}<|eot_id|><|start_header_id|>user<|end_header_id|>

{input}<|eot_id|><|start_header_id|>assistant<|end_header_id|>

{output}<|eot_id|>"""

def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []
    for instruction, input_text, output in zip(instructions, inputs, outputs):
        text = llama3_prompt.format(instruction=instruction, input=input_text, output=output)
        texts.append(text)
    return { "text" : texts }

dataset = dataset.map(formatting_prompts_func, batched = True)
print(f"Loaded {len(dataset)} records. Sample prompt:\n{dataset[0]['text'][:600]}...")


In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4, # Effective batch size = 8
        warmup_steps = 5,
        max_steps = 60, # ~1.2 epochs over 402 records
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

trainer_stats = trainer.train()


In [ ]:
FastLanguageModel.for_inference(model)

sample_input = (
    "Patient reports: Severe throbbing headache on right side, nausea, sensitivity to bright lights.\n"
    "Additional context: severity appears severe; expected/typical duration of care: 3-4 days.\n"
    "Patient profile: 32 years, Female.\n"
    "Lifestyle factors: sleep: irregular sleep, stress: high stress, physical activity: low."
)

prompt = llama3_prompt.format(
    instruction = dataset[0]["instruction"],
    input = sample_input,
    output = ""
)

inputs = tokenizer([prompt], return_tensors = "pt").to("cuda")
outputs = model.generate(**inputs, max_new_tokens = 512, use_cache = True)
response = tokenizer.batch_decode(outputs)[0]

print("=== MODEL OUTPUT CLINICAL JSON ===")
print(response.split("<|start_header_id|>assistant<|end_header_id|>")[-1].replace("<|eot_id|>", "").strip())


In [ ]:
import shutil
from google.colab import files

# Save adapter & tokenizer
model.save_pretrained("medikiosk_llama3_lora")
tokenizer.save_pretrained("medikiosk_llama3_lora")

# Zip and download
shutil.make_archive("medikiosk_llama3_lora", 'zip', "medikiosk_llama3_lora")
files.download("medikiosk_llama3_lora.zip")
print("LoRA adapter archived and downloaded!")
